# Looming Analysis Notebook

This notebook provides tools for loading `.braidz` files and analyzing the angular velocity response of flies to looming stimuli.

In [ ]:
import zipfile
import gzip
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

%matplotlib inline

## Helper Functions

We'll define functions to load data from `.braidz` zip archives and process trajectories.

In [ ]:
def load_braidz(file_path: str):
    """
    Load kalman estimates and stimulus data from a .braidz file.
    """
    with zipfile.ZipFile(file_path, "r") as z:
        # Load kalman_estimates.csv.gz
        with z.open("kalman_estimates.csv.gz") as f:
            with gzip.open(f) as gz:
                df_kalman = pl.read_csv(gz)

        # Load stim.csv if it exists
        df_stim = None
        if "stim.csv" in z.namelist():
            with z.open("stim.csv") as f:
                df_stim = pl.read_csv(f)

    return df_kalman, df_stim


def calculate_angular_velocity(
    xvel: np.ndarray, yvel: np.ndarray, dt: float = 0.01
) -> np.ndarray:
    """
    Calculate angular velocity from velocity components.
    """
    theta = np.arctan2(yvel, xvel)
    theta_unwrap = np.unwrap(theta)
    angular_velocity = np.gradient(theta_unwrap, dt)
    return angular_velocity

## Data Loading

Let's load one of the files and see what it contains.

In [ ]:
file_path = "data/20260407_171532.braidz"
df_kalman, df_stim = load_braidz(file_path)

print(f"Loaded {len(df_kalman)} kalman estimates.")
if df_stim is not None:
    print(f"Loaded {len(df_stim)} stimulus presentations.")
    print(df_stim.head())
else:
    print("No stim.csv found in this file.")

## Trajectory Extraction and Processing

We'll extract trajectories for each stimulus event and calculate the angular velocity response.

In [ ]:
def extract_responses(
    df_kalman: pl.DataFrame, df_stim: pl.DataFrame, window_sec: float = 0.5
):
    """
    Extract response trajectories for each stimulus.
    """
    responses = []
    dt = 0.01  # 100Hz
    n_frames = int(window_sec / dt)

    # To make it efficient, we'll group kalman by obj_id
    # Using a dictionary for fast lookup of object tracks
    # The keys in partition_by(..., as_dict=True) are tuples
    kalman_grouped = df_kalman.partition_by("obj_id", as_dict=True)

    for row in df_stim.iter_rows(named=True):
        obj_id = row["obj_id"]
        start_frame = row["frame"]

        # The key in the partitioned dict is a tuple (obj_id,)
        obj_key = (obj_id,)
        if obj_key in kalman_grouped:
            df_obj = kalman_grouped[obj_key]
            # Filter for the relevant window: from start_frame up to n_frames after
            df_response = df_obj.filter(
                (pl.col("frame") >= start_frame)
                & (pl.col("frame") < start_frame + n_frames)
            )

            # We take what we have, provided it's at least some data points
            if len(df_response) > 5:
                # Calculate angular velocity
                xvel = df_response["xvel"].to_numpy()
                yvel = df_response["yvel"].to_numpy()

                # Optional: Smooth velocities first to reduce noise in angular velocity
                # xvel = savgol_filter(xvel, 7, 3)
                # yvel = savgol_filter(yvel, 7, 3)

                ang_vel = calculate_angular_velocity(xvel, yvel, dt)

                # Store response metadata
                response_data = {
                    "ang_vel": ang_vel,
                    "time": (df_response["frame"].to_numpy() - start_frame) * dt,
                    **{
                        k: v
                        for k, v in row.items()
                        if k not in ["frame", "timestamp", "obj_id"]
                    },
                }
                responses.append(response_data)

    return responses


responses = extract_responses(df_kalman, df_stim, window_sec=1.0)
print(f"Extracted {len(responses)} valid responses.")

## Grouping and Plotting

Now we'll group the responses by stimulus parameters and plot the average angular velocity response.

In [ ]:
def plot_responses(responses, group_by: str = "stimulus_offset_deg"):
    """
    Group responses and plot the mean angular velocity.
    """
    # Grouping
    groups = {}
    for r in responses:
        val = r.get(group_by)
        if val not in groups:
            groups[val] = []
        groups[val].append(r["ang_vel"])

    plt.figure(figsize=(10, 6))

    for label, data_list in groups.items():
        # Pad/truncate to same length for averaging if needed
        max_len = max(len(d) for d in data_list)
        padded_data = []
        for d in data_list:
            if len(d) < max_len:
                padded = np.pad(
                    d, (0, max_len - len(d)), "constant", constant_values=np.nan
                )
            else:
                padded = d[:max_len]
            padded_data.append(padded)

        padded_data = np.stack(padded_data)
        mean_resp = np.nanmean(padded_data, axis=0)
        sem_resp = np.nanstd(padded_data, axis=0) / np.sqrt(
            np.count_nonzero(~np.isnan(padded_data), axis=0)
        )

        time = np.arange(max_len) * 0.01

        (line,) = plt.plot(
            time, mean_resp, label=f"{group_by}={label} (n={len(data_list)})"
        )
        plt.fill_between(time, mean_resp - sem_resp, mean_resp + sem_resp, alpha=0.2)

    plt.xlabel("Time after stimulus start (s)")
    plt.ylabel("Angular velocity (rad/s)")
    plt.title(f"Fly Response to Looming (grouped by {group_by})")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


if responses:
    plot_responses(responses, group_by="stimulus_offset_deg")